# Task 3 — Descriptive Analytics & Product Metrics

**Splendor Analytics Trial Activation Challenge**

This notebook runs descriptive analyses on the trial data and derives commonly used product metrics to inform the product team's decision-making.

## Section 1: Setup & Load

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from scipy import stats

# Load and clean — same steps as Task 1
df = pd.read_csv(r"../data/DA_task.csv")

date_cols = ["TIMESTAMP", "CONVERTED_AT", "TRIAL_START", "TRIAL_END"]
for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors="coerce")

df.columns = df.columns.str.lower()
df = df.drop_duplicates().reset_index(drop=True).copy()
df["days_into_trial"] = (df["timestamp"] - df["trial_start"]).dt.days
df["within_trial"]    = (
    (df["timestamp"] >= df["trial_start"]) &
    (df["timestamp"] <= df["trial_end"])
)
df_clean = df[df["within_trial"]].copy()

print(f"Clean events: {len(df_clean):,}")
print(f"Unique orgs:  {df_clean['organization_id'].nunique()}")

In [ ]:
# Build org-level summary table
activity_counts = (
    df_clean
    .groupby(["organization_id", "activity_name"])
    .size()
    .unstack(fill_value=0)
)

org_stats = (
    df_clean
    .groupby("organization_id")
    .agg(
        converted         = ("converted",       "first"),
        converted_at      = ("converted_at",    "first"),
        trial_start       = ("trial_start",     "first"),
        trial_end         = ("trial_end",       "first"),
        total_events      = ("activity_name",   "count"),
        unique_activities = ("activity_name",   "nunique"),
        days_active       = ("days_into_trial", "nunique"),
        first_event_day   = ("days_into_trial", "min"),
        last_event_day    = ("days_into_trial", "max"),
    )
)

org_df = org_stats.join(activity_counts)

# Apply Trial Goals from Task 1
org_df["goal1_met"] = (org_df.get("Scheduling.Shift.Created",               0) >= 3).astype(int)
org_df["goal2_met"] = (org_df.get("Scheduling.Template.ApplyModal.Applied",  0) >= 1).astype(int)
org_df["goal3_met"] = (org_df.get("PunchClock.PunchedIn",                    0) >= 1).astype(int)
org_df["trial_activated"] = (
    (org_df["goal1_met"] == 1) &
    (org_df["goal2_met"] == 1) &
    (org_df["goal3_met"] == 1)
).astype(int)

org_df["trial_month"] = org_df["trial_start"].dt.to_period("M").astype(str)

print(f"Org table shape: {org_df.shape}")
print(f"\nTrial cohorts:\n{org_df['trial_month'].value_counts().sort_index()}")

## Section 2: Conversion Rate

In [ ]:
total_orgs     = len(org_df)
converted_orgs = org_df["converted"].sum()
conv_rate      = converted_orgs / total_orgs * 100

print("── Overall Conversion Rate ──────────────────────")
print(f"Total organisations:     {total_orgs}")
print(f"Converted organisations: {converted_orgs}")
print(f"Conversion rate:         {conv_rate:.1f}%")

In [ ]:
# Conversion rate by cohort month
cohort_conv = (
    org_df.groupby("trial_month")
    .agg(
        total_orgs     = ("converted", "count"),
        converted_orgs = ("converted", "sum"),
    )
    .assign(conversion_rate = lambda x: (x["converted_orgs"] / x["total_orgs"] * 100).round(1))
)

print("── Conversion Rate by Cohort Month ──────────────")
print(cohort_conv.to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(cohort_conv.index, cohort_conv["total_orgs"],
            color=["#5b8dd9","#6aaf50","#d4a84b"], edgecolor="none")
axes[0].set_title("Organisations per Cohort", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Trial Start Month")
axes[0].set_ylabel("Number of Organisations")
for i,(idx,row) in enumerate(cohort_conv.iterrows()):
    axes[0].text(i, row["total_orgs"]+5, str(row["total_orgs"]), ha="center", fontsize=11, fontweight="bold")

axes[1].bar(cohort_conv.index, cohort_conv["conversion_rate"],
            color=["#5b8dd9","#6aaf50","#d4a84b"], edgecolor="none")
axes[1].axhline(y=conv_rate, color="red", linestyle="--", linewidth=1.5, label=f"Overall avg ({conv_rate:.1f}%)")
axes[1].set_title("Conversion Rate by Cohort", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Trial Start Month")
axes[1].set_ylabel("Conversion Rate (%)")
axes[1].legend()
axes[1].set_ylim(0, 35)
for i,(idx,row) in enumerate(cohort_conv.iterrows()):
    axes[1].text(i, row["conversion_rate"]+0.5, f"{row['conversion_rate']}%", ha="center", fontsize=11, fontweight="bold")

plt.suptitle("Conversion Overview by Trial Cohort", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("conversion_by_cohort.png", bbox_inches="tight", dpi=150)
plt.show()

## Section 3: Time to Convert

In [ ]:
converted_df = org_df[org_df["converted"] == True].copy()
converted_df["days_to_convert"] = (converted_df["converted_at"] - converted_df["trial_start"]).dt.days

print("── Time to Convert (days) ───────────────────────")
print(f"Converted orgs:  {len(converted_df)}")
print(f"Median:          {converted_df['days_to_convert'].median():.0f} days")
print(f"Mean:            {converted_df['days_to_convert'].mean():.1f} days")
print(f"Min:             {converted_df['days_to_convert'].min():.0f} days")
print(f"Max:             {converted_df['days_to_convert'].max():.0f} days")

In [ ]:
def time_band(days):
    if days <= 7:    return "0-7 days"
    elif days <= 14: return "8-14 days"
    elif days <= 21: return "15-21 days"
    elif days <= 30: return "22-30 days"
    else:            return "30+ days"

converted_df["time_band"] = converted_df["days_to_convert"].apply(time_band)
band_order  = ["0-7 days","8-14 days","15-21 days","22-30 days","30+ days"]
band_counts = converted_df["time_band"].value_counts().reindex(band_order).fillna(0).astype(int)
band_pct    = (band_counts / len(converted_df) * 100).round(1)

print("── Conversions by Time Band ─────────────────────")
for band, count, pct in zip(band_order, band_counts, band_pct):
    print(f"  {band:<14}  {count:>3} orgs  ({pct}%)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(converted_df["days_to_convert"].dropna(), bins=30,
             color="#5b8dd9", edgecolor="white", linewidth=0.5)
axes[0].axvline(converted_df["days_to_convert"].median(), color="#e06060",
                linestyle="--", linewidth=1.8,
                label=f"Median: {converted_df['days_to_convert'].median():.0f} days")
axes[0].set_title("Distribution of Days to Convert", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Days from Trial Start to Conversion")
axes[0].set_ylabel("Number of Organisations")
axes[0].legend()

axes[1].bar(band_order, band_counts.values, color="#6aaf50", edgecolor="none")
for i,(count,pct) in enumerate(zip(band_counts.values, band_pct.values)):
    axes[1].text(i, count+1, f"{count}\n({pct}%)", ha="center", fontsize=10, fontweight="bold")
axes[1].set_title("Conversions by Time Band", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Days into Trial")
axes[1].set_ylabel("Number of Organisations")

plt.suptitle("How Long Does It Take Orgs to Convert?", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("time_to_convert.png", bbox_inches="tight", dpi=150)
plt.show()

## Section 4: Trial Goal Funnel

In [ ]:
total          = len(org_df)
goal1_orgs     = org_df["goal1_met"].sum()
goal2_orgs     = org_df["goal2_met"].sum()
goal3_orgs     = org_df["goal3_met"].sum()
activated_orgs = org_df["trial_activated"].sum()

funnel_data = {
    "Started Trial":          total,
    "Goal 1 — Shift Created": goal1_orgs,
    "Goal 2 — Template Used": goal2_orgs,
    "Goal 3 — Punched In":    goal3_orgs,
    "Trial Activated":        activated_orgs,
}

print(f"{'Stage':<30} {'Orgs':>6}  {'% of Total':>10}  {'Drop-off':>10}")
print("─" * 62)
prev = total
for stage, count in funnel_data.items():
    pct     = count / total * 100
    dropoff = prev - count
    drop_pct= dropoff / prev * 100 if prev > 0 else 0
    print(f"{stage:<30} {count:>6}  {pct:>9.1f}%  {f'-{dropoff} ({drop_pct:.1f}%)':>12}")
    prev = count

In [ ]:
stages = list(funnel_data.keys())
counts = list(funnel_data.values())
pcts   = [c / total * 100 for c in counts]
colors = ["#4a7fc1","#5b8dd9","#6aaf50","#d4a84b","#e06060"]

fig, ax = plt.subplots(figsize=(13, 7))
bars = ax.barh(stages[::-1], counts[::-1], color=colors[::-1], edgecolor="none", height=0.55)

for bar, count, pct in zip(bars, counts[::-1], pcts[::-1]):
    ax.text(bar.get_width()-15, bar.get_y()+bar.get_height()/2,
            f"{count:,}  ({pct:.1f}%)", va="center", ha="right",
            fontsize=12, fontweight="bold", color="white")

ax.set_xlabel("Number of Organisations", fontsize=12)
ax.set_title("Trial Activation Funnel\nHow many orgs completed each goal?",
             fontsize=14, fontweight="bold")
ax.set_xlim(0, total*1.05)
ax.spines[["top","right","left"]].set_visible(False)
ax.tick_params(axis="y", labelsize=12)
plt.tight_layout()
plt.savefig("trial_goal_funnel.png", bbox_inches="tight", dpi=150)
plt.show()

In [ ]:
org_df["goals_completed"] = org_df["goal1_met"] + org_df["goal2_met"] + org_df["goal3_met"]
overlap     = org_df["goals_completed"].value_counts().sort_index()
overlap_pct = (overlap / total * 100).round(1)

labels = {0:"No goals", 1:"1 goal only", 2:"2 goals", 3:"All 3 goals (Activated)"}
print("── Goals Completed per Organisation ─────────────")
for k, count in overlap.items():
    print(f"  {labels[k]:<28} {count:>4} orgs  ({overlap_pct[k]}%)")

## Section 5: Feature & Activity Adoption

In [ ]:
activity_cols = [col for col in org_df.columns
                 if any(col.startswith(x) for x in [
                     "Scheduling","PunchClock","Absence",
                     "Timesheets","Communication","Mobile",
                     "Shift","Break","Revenue","Integration"])]

adoption = (org_df[activity_cols] > 0).mean() * 100
adoption = adoption.round(1).sort_values(ascending=False)

adoption_df = pd.DataFrame({
    "activity":      adoption.index,
    "adoption_rate": adoption.values,
    "orgs_used":     (org_df[activity_cols] > 0).sum().values,
})

print("── Activity Adoption Rates ──────────────────────")
print(adoption_df.to_string(index=False))

In [ ]:
colors = ["#6aaf50" if r>=40 else "#d4a84b" if r>=15 else "#e06060"
          for r in adoption_df["adoption_rate"]]

fig, ax = plt.subplots(figsize=(13, 10))
bars = ax.barh(adoption_df["activity"][::-1], adoption_df["adoption_rate"][::-1],
               color=colors[::-1], edgecolor="none", height=0.65)
for bar, rate in zip(bars, adoption_df["adoption_rate"][::-1]):
    ax.text(bar.get_width()+0.5, bar.get_y()+bar.get_height()/2,
            f"{rate}%", va="center", ha="left", fontsize=10, fontweight="bold")

from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(facecolor="#6aaf50", label="High adoption (>= 40%)"),
    Patch(facecolor="#d4a84b", label="Medium adoption (15-39%)"),
    Patch(facecolor="#e06060", label="Low adoption (< 15%)"),
], loc="lower right", fontsize=10)

ax.set_xlabel("% of Organisations That Used This Activity", fontsize=12)
ax.set_title("Feature Adoption During Trial", fontsize=14, fontweight="bold")
ax.set_xlim(0, 105)
ax.spines[["top","right","left"]].set_visible(False)
plt.tight_layout()
plt.savefig("feature_adoption.png", bbox_inches="tight", dpi=150)
plt.show()

In [ ]:
module_map = {
    "Scheduling":    [c for c in activity_cols if c.startswith("Scheduling")],
    "PunchClock":    [c for c in activity_cols if c.startswith("PunchClock")],
    "Absence":       [c for c in activity_cols if c.startswith("Absence")],
    "Timesheets":    [c for c in activity_cols if c.startswith("Timesheets")],
    "Communication": [c for c in activity_cols if c.startswith("Communication")],
}

print("── Module-Level Adoption ────────────────────────")
print(f"{'Module':<16} {'Orgs Used':>10}  {'Adoption Rate':>14}")
print("─" * 44)
for module, cols in module_map.items():
    used = (org_df[cols] > 0).any(axis=1).sum()
    rate = used / total * 100
    print(f"{module:<16} {used:>10}  {rate:>13.1f}%")

## Section 6: Engagement Depth

In [ ]:
engagement_metrics = ["total_events","unique_activities","days_active","first_event_day","last_event_day"]

print("── Engagement Depth by Conversion Status ────────")
for metric in engagement_metrics:
    conv_med  = org_df[org_df["converted"]==True][metric].median()
    nconv_med = org_df[org_df["converted"]==False][metric].median()
    print(f"\n  {metric}")
    print(f"    Converters     — median: {conv_med:>6.1f}")
    print(f"    Non-converters — median: {nconv_med:>6.1f}")

In [ ]:
org_df["trial_length"] = (org_df["trial_end"] - org_df["trial_start"]).dt.days + 1
org_df["stickiness"]   = (org_df["days_active"] / org_df["trial_length"]).round(3)

print("── Stickiness Score (days_active / trial_length) ─")
print(f"  Overall median:         {org_df['stickiness'].median():.3f}")
print(f"  Converters median:      {org_df[org_df['converted']==True]['stickiness'].median():.3f}")
print(f"  Non-converters median:  {org_df[org_df['converted']==False]['stickiness'].median():.3f}")
print(f"  Very sticky  (>= 0.5): {(org_df['stickiness']>=0.5).sum()} orgs ({(org_df['stickiness']>=0.5).mean()*100:.1f}%)")
print(f"  Moderate (0.2-0.5):    {org_df['stickiness'].between(0.2,0.5).sum()} orgs ({org_df['stickiness'].between(0.2,0.5).mean()*100:.1f}%)")
print(f"  Low (< 0.2):           {(org_df['stickiness']<0.2).sum()} orgs ({(org_df['stickiness']<0.2).mean()*100:.1f}%)")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()
plots = [
    ("total_events",      "Total Events per Org"),
    ("unique_activities", "Unique Activities per Org"),
    ("days_active",       "Days Active During Trial"),
    ("stickiness",        "Stickiness Score"),
]
for ax, (metric, title) in zip(axes, plots):
    for status, color, label in [(True,"#6aaf50","Converted"),(False,"#e06060","Not Converted")]:
        subset = org_df[org_df["converted"]==status][metric].dropna()
        ax.hist(subset, bins=30, alpha=0.6, color=color,
                label=f"{label} (n={len(subset)})", edgecolor="none")
    ax.set_title(title, fontsize=12, fontweight="bold")
    ax.set_ylabel("Organisations")
    ax.legend(fontsize=9)
    ax.spines[["top","right"]].set_visible(False)

plt.suptitle("Engagement Depth — Converted vs Non-Converted",
             fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("engagement_depth.png", bbox_inches="tight", dpi=150)
plt.show()

## Section 7: Product Metrics Dashboard

In [ ]:
activation_rate      = org_df["trial_activated"].mean() * 100
goal1_rate           = org_df["goal1_met"].mean() * 100
goal2_rate           = org_df["goal2_met"].mean() * 100
goal3_rate           = org_df["goal3_met"].mean() * 100
activated_mask       = org_df["trial_activated"] == 1
activation_conv_rate = org_df[activated_mask]["converted"].mean() * 100
non_activation_conv  = org_df[~activated_mask]["converted"].mean() * 100
org_df["active_final_week"] = (org_df["last_event_day"] >= 23).astype(int)
final_week_rate      = org_df["active_final_week"].mean() * 100

print("=" * 58)
print("  PRODUCT METRICS DASHBOARD")
print("=" * 58)
print(f"\n  ACTIVATION")
print(f"  Trial Activation Rate          {activation_rate:.1f}%")
print(f"  Goal 1 Completion Rate         {goal1_rate:.1f}%")
print(f"  Goal 2 Completion Rate         {goal2_rate:.1f}%")
print(f"  Goal 3 Completion Rate         {goal3_rate:.1f}%")
print(f"\n  CONVERSION")
print(f"  Overall Conversion Rate        {conv_rate:.1f}%")
print(f"  Activated -> Converted         {activation_conv_rate:.1f}%")
print(f"  Not Activated -> Converted     {non_activation_conv:.1f}%")
print(f"\n  ENGAGEMENT")
print(f"  Avg Unique Activities / Org    {org_df['unique_activities'].mean():.1f}")
print(f"  Avg Events / Org               {org_df['total_events'].mean():.0f}")
print(f"\n  RETENTION PROXY")
print(f"  Active in Final Week (day 23+) {final_week_rate:.1f}%")
print(f"  Overall Stickiness Median      {org_df['stickiness'].median():.3f}")
print("=" * 58)

## Section 8: Recommendations

In [ ]:
recommendations = """
================================================================
  PRODUCT TEAM RECOMMENDATIONS
  Based on Trial Activation Analysis — 966 Orgs
================================================================

1. FIX THE GOAL 2 BOTTLENECK
   58.3% of orgs hit Goal 1 but only 11.2% reach Goal 2.
   That is an 80% drop-off between two features in the same
   module. Add an in-app nudge after the 3rd shift is created.

2. PUSH PUNCHCLOCK ACTIVATION EARLIER
   Only 21.8% of orgs ever had a team member clock in.
   Build a guided invite-your-team step into onboarding.

3. INVESTIGATE THE MARCH COHORT DROP
   Conversion fell from 23% in Jan/Feb to 18.2% in March.
   Cross-reference with sales and product changes that month.

4. DOUBLE DOWN ON SCHEDULING AS THE ANCHOR MODULE
   88.2% adoption. Design onboarding as Scheduling-first,
   then expand naturally to PunchClock and Communication.

5. STUDY THE 95 HIGHLY STICKY ORGS
   They represent the product's power users. Their behaviour
   is the blueprint for ideal trial engagement.

6. REFINE TRIAL GOALS QUARTERLY
   As more cohorts mature, rerun this analysis to tighten
   goal thresholds and track whether the conversion gap grows.
"""

print(recommendations)

In [ ]:
# Export final enriched org metrics table
task3_cols = [
    "converted","converted_at","trial_start","trial_end",
    "trial_month","total_events","unique_activities",
    "days_active","first_event_day","last_event_day",
    "trial_length","stickiness","goals_completed",
    "goal1_met","goal2_met","goal3_met","trial_activated",
    "active_final_week",
]

org_df[task3_cols].to_csv("task3_org_metrics.csv", index=True)
print("task3_org_metrics.csv exported successfully.")
print(f"Shape: {org_df[task3_cols].shape}")